# 2 — Selecting subtype markers

**Needs SCimilarity `model_v1.1`, a GPU, and an atlas. Runs in about a minute once the
model is loaded.**

[Tutorial 1](01_quickstart) used a stub encoder on planted data, where the right answer was known
in advance. This notebook does the same thing for real: four sibling cytotoxic T-cell subtypes
from a human immune atlas, a real foundation model, and no ground truth to fall back on.

Two things here are not in the quickstart, and both are the kind of mistake that produces a
plausible-looking panel rather than an error message:

1. **Gene-space alignment** — the encoder has its own gene universe and its own ordering.
2. **Who normalizes what** — `attribute()` hands `adata.X` to *two* consumers with *opposite*
   input requirements. Getting this wrong is silent, and the contrast QC will not catch it. We
   demonstrate that at the end rather than asserting it.

Along the way we run the comparison that motivates the method: the same four subtypes ranked by a
marginal log-fold-change, on exactly the same cells.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import time, itertools
import numpy as np, pandas as pd, anndata as ad, scipy.sparse as sp
import focal
from focal.encoders import SCimilarityEncoder, Encoder
from focal.qc import contrast_qc
from scimilarity.utils import align_dataset

# --- point these at your own copies -------------------------------------------------
MODEL = "/data/gli9/Jian/sc_age_clock/models/scimilarity_model/model_v1.1"
H5AD  = "/data/gli9/test_sig/scattr_benchmark/phase2/dominguez_conde_fullgene.h5ad"
# ------------------------------------------------------------------------------------

print("focal", focal.__version__)

focal 0.5.0


## The data, and why it is a *subset*

The atlas is the cross-tissue human immune atlas of Domínguez Conde et al. (2022); this file is
the gene-complete version used in the FOCAL manuscript, with raw integer counts in `.X`.

We keep only the four cytotoxic T subtypes. That is not a convenience — it *is* the query.
`reference="siblings"` means *the other cells in the object you passed*, so the object boundary
**is** the biological alternative you are asking about. Hand FOCAL the whole atlas and you ask
"what makes Temra cells different from monocytes and B cells"; the answer would be the CD8 T-cell
program, not the Temra program. Subsetting to a lineage is how the subtype question gets asked.

In [2]:
SIBS = ["Tem/Temra cytotoxic T cells", "Tcm/Naive cytotoxic T cells",
        "Tem/Trm cytotoxic T cells", "gdT"]

A = ad.read_h5ad(H5AD)
A = A[A.obs["label"].isin(SIBS)].copy()
A.obs["label"] = A.obs["label"].cat.remove_unused_categories()

X0 = A.X[:200].toarray() if sp.issparse(A.X) else np.asarray(A.X[:200])
print("subset:", A.shape, "| .X holds raw integer counts:", np.allclose(X0, np.round(X0)))
print(A.obs["label"].value_counts().to_string())

subset: (3750, 47332) | .X holds raw integer counts: True
label
Tem/Temra cytotoxic T cells    1759
Tcm/Naive cytotoxic T cells     957
Tem/Trm cytotoxic T cells       618
gdT                             416


## Contract 1 — gene space

`SCimilarityEncoder` is a fixed-input network: it expects exactly the genes listed in the model's
`gene_order.tsv`, in that order. `align_dataset` reindexes your object onto that universe —
dropping genes the model does not know, and inserting all-zero columns for model genes your data
lacks.

In [3]:
go = pd.read_csv(f"{MODEL}/gene_order.tsv", header=None)[0].tolist()
print(f"model gene space: {len(go)} genes | present in this atlas: {len(set(go) & set(A.var_names))}")
print(f"atlas genes the model does not use: {len(set(A.var_names) - set(go))}")

A = align_dataset(A, go)
Xa = A.X[:200].toarray() if sp.issparse(A.X) else np.asarray(A.X[:200])
print("aligned:", A.shape, "| still raw integer counts:", np.allclose(Xa, np.round(Xa)))

model gene space: 28231 genes | present in this atlas: 28053
atlas genes the model does not use: 19279


aligned: (3750, 28231) | still raw integer counts: True


Note the second half of that print. **Alignment must not normalize.** Here is why.

## Contract 2 — `adata.X` has two consumers, and they disagree

Read the docstrings and you will find what looks like a contradiction:

- `SCimilarityEncoder.embed()` documents that it needs *per-cell tp10k-log-normalized* input and
  "does not normalize internally; callers must preprocess".
- `focal.centroid.mean_lognorm_centroid` documents that it takes *raw counts* — it applies the
  per-cell tp10k-lognorm itself, then averages.

Both are true, and inside `attribute()` both are handed the **same** `adata.X`:

```python
Z_t, Z_r = enc.embed(counts[target_mask]), enc.embed(counts[ref_mask])   # wants lognorm
C        = mean_lognorm_centroid(counts, target_mask)                    # wants raw counts
...
ig.attribute(x=C, ...)  ->  enc.torch_encode(C)                          # C is ALREADY lognorm
```

So exactly one arrangement satisfies every consumer:

| | receives | therefore must |
|---|---|---|
| `adata.X` | — | stay **raw counts** |
| `.embed()` | raw counts | **normalize inside** |
| `.torch_encode()` | the centroid `C`, already log-normalized | **pass straight through** |

That asymmetry — normalize in `.embed()`, do *not* normalize in `.torch_encode()` — is the whole
trick, and it is why using SCimilarity with `attribute()` goes through an eight-line encoder
wrapper — the same one the repository's own `reproduce/reproduce_focal.py` defines.

In [4]:
class PerCellLognormEncoder(Encoder):
    '''Let a SCimilarity encoder accept gene-aligned RAW counts from focal.attribute().

    .embed() lognorms, because attribute() feeds it adata.X (raw counts).
    .torch_encode() must NOT, because attribute() feeds it the reference centroid, which
    mean_lognorm_centroid has already log-normalized.'''

    def __init__(self, base_encoder):
        self._base = base_encoder

    def embed(self, counts):
        X = counts.toarray() if sp.issparse(counts) else np.asarray(counts)
        X = X.astype("float32")
        totals = X.sum(axis=1, keepdims=True)
        totals[totals == 0] = 1.0
        return self._base.embed(np.log1p(1e4 * X / totals).astype("float32"))

    def torch_encode(self, x):
        return self._base.torch_encode(x)


base = SCimilarityEncoder(MODEL, device="cuda")
enc  = PerCellLognormEncoder(base)

:::{warning}
`attribute(..., device="cuda")` does not move the encoder's weights — it only selects where the
attribution runs. We passed `device="cuda"` to `SCimilarityEncoder` above so the network is
already there. Pass a device the model does not live on and you get a device error, not a silent
fallback.
:::

## Running it

`target=None` attributes every subtype in turn, each against its own siblings.

In [5]:
t0 = time.time()
res = focal.attribute(enc, A, "label", reference="siblings", device="cuda")
print(f"4 subtypes attributed in {time.time() - t0:.1f} s\n")

for s in A.obs["label"].cat.categories:
    print(f"{s:32s} {res.top(s, 10)}")

4 subtypes attributed in 14.1 s

Tcm/Naive cytotoxic T cells      ['IL7R', 'VIM', 'CCR7', 'NOSIP', 'LTB', 'TRAC', 'RGS10', 'KLF2', 'RPS4Y1', 'TCF7']
Tem/Temra cytotoxic T cells      ['NKG7', 'GNLY', 'KLRD1', 'GZMH', 'CTSW', 'CST7', 'GZMA', 'PRF1', 'GZMB', 'FGFBP2']
Tem/Trm cytotoxic T cells        ['GZMK', 'CD8B', 'CD8A', 'DUSP2', 'CMC1', 'IL32', 'CST7', 'GZMA', 'LIME1', 'TRAC']
gdT                              ['TYROBP', 'TRDC', 'KLRB1', 'GNLY', 'CTSW', 'KLRC1', 'NKG7', 'CD247', 'HOPX', 'PRF1']


These are recognisable panels, and nothing supplied a ground truth to get them:

- **Tcm/Naive** — `IL7R`, `CCR7`, `TCF7`, `KLF2`, `LTB`: the canonical naive / central-memory
  program.
- **Tem/Temra** — `NKG7`, `GNLY`, `PRF1`, `GZMB`, `GZMH`, `FGFBP2`: terminal cytotoxic effectors.
- **Tem/Trm** — `GZMK` first, with `CD8A`/`CD8B`: the GZMK-high effector-memory population.
- **gdT** — `TRDC` and `KLRB1`. `TRDC` is the T-cell receptor δ constant region, the defining
  locus of a γδ T cell, recovered without anything having been told that γδ is what sets this
  group apart.

## What a marginal ranking gives you on the same cells

The honest comparison holds everything fixed — same cells, same gene space, same labels — and
changes only the *statistic*: mean log-normalized expression in the subtype minus mean in its
siblings.

In [6]:
Xd = A.X.toarray() if sp.issparse(A.X) else np.asarray(A.X)
L  = np.log1p(1e4 * Xd / np.maximum(Xd.sum(1, keepdims=True), 1)).astype("float32")
lab = A.obs["label"].values
ribo = lambda gs: sum(g.startswith(("RPL", "RPS")) for g in gs)

rows = []
for s in A.obs["label"].cat.categories:
    m = lab == s
    lfc = pd.Series(L[m].mean(0) - L[~m].mean(0), index=A.var_names).sort_values(ascending=False)
    top_lfc, top_focal = list(lfc.index[:10]), res.top(s, 10)
    print(f"{s}\n  logFC: {top_lfc}\n")
    rows.append({"subtype": s.replace(" cytotoxic T cells", ""),
                 "shared@10": len(set(top_lfc) & set(top_focal)),
                 "ribosomal, logFC": ribo(top_lfc), "ribosomal, FOCAL": ribo(top_focal)})

pd.DataFrame(rows).set_index("subtype")

Tcm/Naive cytotoxic T cells
  logFC: ['IL7R', 'RPL23', 'LINC02446', 'EIF3E', 'RPS10', 'NOSIP', 'RPL7', 'RPS20', 'CCR7', 'EEF1B2']



Tem/Temra cytotoxic T cells
  logFC: ['GNLY', 'NKG7', 'CCL5', 'GZMH', 'CST7', 'GZMB', 'GZMA', 'FGFBP2', 'KLRD1', 'S100A4']



Tem/Trm cytotoxic T cells
  logFC: ['GZMK', 'CMC1', 'CCL5', 'DUSP2', 'CD74', 'COTL1', 'GZMA', 'CST7', 'CD69', 'GPR183']



gdT
  logFC: ['TRDC', 'GNLY', 'TYROBP', 'KLRB1', 'TRGC1', 'KLRC1', 'KLRD1', 'TRDV2', 'NKG7', 'HOPX']



,shared@10,"ribosomal, logFC","ribosomal, FOCAL"
subtype,,,
Tcm/Naive,3,4,1
Tem/Temra,8,0,0
Tem/Trm,5,0,0
gdT,7,0,0


The two rankings agree where you would expect and diverge where it matters.

For **Tem/Temra**, agreement is high — a population defined by a handful of very highly expressed
granzymes and perforin is exactly the regime where a marginal test already works. FOCAL is not
better here, and the paper says so.

For **Tcm/Naive**, agreement collapses, and the disagreement has a direction: four of the
log-fold-change list's ten genes are ribosomal proteins, and two more (`EIF3E`, `EEF1B2`) are
translation factors that the crude `RPL*`/`RPS*` rule does not even catch — so six of ten are
translation machinery. Naive T cells really do carry higher ribosomal content than effector
cells, so these are not a statistical artefact. They are a true difference that is useless as an
identity program, because it is shared by every naive lymphocyte in the body. This is [Tutorial
1](01_quickstart)'s housekeeping trap, unplanted, on real data.

FOCAL's list scores 1 on the same rule, and it is worth saying what that one gene is rather than
rounding it to zero: `RPS4Y1` is Y-linked, so it is matched by the `RPS` prefix but is really a
donor-sex signal — a *different* confound, and one FOCAL did not remove. Attribution suppresses
genes that do not move along the contrast; it does not know which of the genes that do move you
consider biology.

## Contrast QC on a real contrast

In [7]:
res.qc.round(3)

,n_target,n_reference,dprime,cos_u_mean,cos_u_min
Tcm/Naive cytotoxic T cells,957.0,2793.0,6.606,0.999,0.999
Tem/Temra cytotoxic T cells,1759.0,1991.0,1.557,0.997,0.993
Tem/Trm cytotoxic T cells,618.0,3132.0,2.554,0.993,0.977
gdT,416.0,3334.0,2.222,0.991,0.958


All four directions are stable (`cos_u` ≈ 0.99), so every panel above describes a reproducible
axis. But separation varies a lot — `Tcm/Naive` is the most separable subtype by a wide margin,
and `Tem/Temra` the least, even though Temra produced the *most* confident-looking marker list.

That is worth pinning down rather than hand-waving. `reference` accepts an explicit list of
labels, which makes every pairwise contrast directly measurable:

In [8]:
pw = pd.DataFrame(np.nan, index=SIBS, columns=SIBS)
for a, b in itertools.permutations(SIBS, 2):
    pw.loc[a, b] = float(contrast_qc(enc, A, "label", target=a, reference=[b])["dprime"].iloc[0])

short = lambda s: s.replace(" cytotoxic T cells", "")
pw.rename(index=short, columns=short).round(2)

,Tem/Temra,Tcm/Naive,Tem/Trm,gdT
Tem/Temra,NaN,6.80,2.38,1.88
Tcm/Naive,6.80,NaN,6.21,8.76
Tem/Trm,2.38,6.21,NaN,3.45
gdT,1.88,8.76,3.45,NaN


There is the explanation. `Tcm/Naive` sits far from all three effector populations, while the
three cytotoxic populations sit close to *each other*. Temra's pooled-sibling reference is
therefore mostly cells that share its cytotoxic program, which compresses the separation. A lower
`d'` here reflects a genuinely harder question, not a defect in the data or the cluster.

This is the most important thing to take from the notebook: **`d'` is a property of the comparison
you chose, not a score for your cluster.** Change the reference and it changes — which is
[Tutorial 3](03_choosing_the_reference).

## The trap, demonstrated

`SCimilarityEncoder`'s own docstring tells you to preprocess with
`scimilarity.utils.lognorm_counts`, which writes the log-normalized matrix into `.X`. Follow that
instruction before calling `attribute()` and `.X` is no longer raw counts, so
`mean_lognorm_centroid` log-normalizes an already-log-normalized matrix. Nothing raises.

`L` above is exactly what `lognorm_counts` would have put in `.X`, so we can run that path and
measure the cost instead of guessing at it.

In [9]:
A_wrong = A.copy()
A_wrong.X = L                      # what lognorm_counts(adata) leaves in .X
wrong = focal.attribute(base, A_wrong, "label", reference="siblings", device="cuda", qc="silent")

comp = pd.DataFrame([{"subtype": short(s),
                      "top-10 kept": len(set(res.top(s, 10)) & set(wrong.top(s, 10))),
                      "top-50 kept": len(set(res.top(s, 50)) & set(wrong.top(s, 50)))}
                     for s in A.obs["label"].cat.categories]).set_index("subtype")
print(comp.to_string(), "\n")
print("QC identical between the correct and the double-normalized path:",
      bool(np.allclose(res.qc[["dprime", "cos_u_mean"]].values,
                       wrong.qc[["dprime", "cos_u_mean"]].values)))

           top-10 kept  top-50 kept
subtype                            
Tcm/Naive            8           39
Tem/Temra           10           43
Tem/Trm              9           42
gdT                  9           42 

QC identical between the correct and the double-normalized path: True


:::{warning}
Read those two outputs together, because the second is the dangerous one.

The panels are *mostly* preserved — this is a partial reshuffle, not a wholesale corruption. Too
small to notice by eye, and large enough to move which genes make a top-*k* cut.

And `res.qc` is **identical** between the two paths. That is not luck: QC is computed from
embeddings alone, and `.embed()` receives correctly-normalized input either way — only the
centroid is damaged. So the diagnostic that exists to tell you a contrast is untrustworthy will
sit there reporting a healthy contrast while the ranking quietly drifts.

**Contrast QC diagnoses the contrast, not your preprocessing.** Keep `.X` raw and put the
normalization inside the encoder.
:::

## Where to go next

- [Tutorial 3](03_choosing_the_reference) — we just *measured* that the reference determines the
  answer. Tutorial 3 uses that on purpose, moving the same target between a fine-subtype and a
  broad-identity explanation by changing one argument.
- [Tutorial 4](04_contrast_qc) — what these QC numbers look like when the contrast is not real.